### Task 1: Select the Best Mission: Game Tree Challenge

In [ ]:
mission_tree = {
    'A': ['B', 'C'], 'B': ['D', 'E'], 'C': ['F', 'G'],
    'D': 4, 'E': 7, 'F': 2, 'G': 9
}

def mission_minimax(node, is_max):
    if isinstance(mission_tree[node], int):
        return mission_tree[node]
    values = [mission_minimax(child, not is_max) for child in mission_tree[node]]
    return max(values) if is_max else min(values)

node_values = {node: mission_minimax(node, node == 'A') for node in ['B', 'C', 'A']}
best_branch = max(mission_tree['A'], key=lambda node: node_values[node])
for node in ['B', 'C', 'A']:
    print(f'{node} = {node_values[node]}')
print(f'MAX selects branch {best_branch} with utility {node_values[best_branch]}')

B = 4
C = 2
A = 4
MAX selects branch B with utility 4


### Task 2: Smart Tic-Tac-Toe Opponent

In [ ]:
import copy

board = [
    ['X', 'O', 'X'],
    [' ', 'X', ' '],
    ['O', ' ', ' ']
]
COMPUTER, HUMAN = 'O', 'X'

def display_board(board):
    for row in board:
        print(' | '.join(cell if cell != ' ' else '.' for cell in row))
    print()

def check_winner(board):
    lines = board + [[board[r][c] for r in range(3)] for c in range(3)]
    lines += [[board[i][i] for i in range(3)], [board[i][2-i] for i in range(3)]]
    for line in lines:
        if line[0] != ' ' and all(cell == line[0] for cell in line):
            return line[0]
    return None

def is_terminal(board):
    return ttt_check_winner(board) is not None or all(cell != ' ' for row in board for cell in row)

def utility(board):
    winner = ttt_check_winner(board)
    return 1 if winner == COMPUTER else -1 if winner == HUMAN else 0

def get_successors(board, player):
    successors = []
    for row in range(3):
        for col in range(3):
            if board[row][col] == ' ':
                next_board = copy.deepcopy(board)
                next_board[row][col] = player
                successors.append(((row, col), next_board))
    return successors

def minimax(board, player):
    if ttt_is_terminal(board):
        return ttt_utility(board)
    next_player = HUMAN if player == COMPUTER else COMPUTER
    values = [ttt_minimax(next_board, next_player) for _, next_board in ttt_get_successors(board, player)]
    return max(values) if player == COMPUTER else min(values)

def get_best_move(board):
    if ttt_is_terminal(board):
        return None, board, ttt_utility(board)
    choices = [(ttt_minimax(next_board, HUMAN), move, next_board)
               for move, next_board in ttt_get_successors(board, COMPUTER)]
    score, move, next_board = max(choices, key=lambda choice: choice[0])
    return move, next_board, score

ttt_display_board = display_board
ttt_check_winner = check_winner
ttt_is_terminal = is_terminal
ttt_utility = utility
ttt_get_successors = get_successors
ttt_minimax = minimax
ttt_get_best_move = get_best_move

print('Initial board:')
display_board(board)
move, updated_board, score = get_best_move(board)
print(f'Computer chooses row {move[0] + 1}, column {move[1] + 1}; expected utility {score}')
display_board(updated_board)

Initial board:
X | O | X
. | X | .
O | . | .

Computer chooses row 3, column 3; expected utility 0
X | O | X
. | X | .
O | . | O



#### Bonus: Play against the computer
Run `play_tic_tac_toe(updated_board)` to enter your own moves.

In [ ]:
def play_tic_tac_toe(start_board, human_moves=None):
    game = copy.deepcopy(start_board)
    scripted = iter(human_moves) if human_moves is not None else None
    while not ttt_is_terminal(game):
        while True:
            try:
                if scripted is None:
                    row = int(input('Human row (1-3): ')) - 1
                    col = int(input('Human column (1-3): ')) - 1
                else:
                    row, col = next(scripted)
            except (ValueError, StopIteration):
                if scripted is not None:
                    raise ValueError('Scripted human moves ran out or contain an invalid move')
                print('Enter row and column numbers from 1 to 3.')
                continue
            if 0 <= row < 3 and 0 <= col < 3 and game[row][col] == ' ':
                break
            print('That cell is unavailable; choose another.')
        game[row][col] = HUMAN
        print(f'Human plays row {row + 1}, column {col + 1}:')
        ttt_display_board(game)
        if ttt_is_terminal(game):
            break
        move, game, score = ttt_get_best_move(game)
        print(f'Computer plays row {move[0] + 1}, column {move[1] + 1} (utility {score}):')
        ttt_display_board(game)
    winner = ttt_check_winner(game)
    print('Computer wins!' if winner == COMPUTER else 'Human wins!' if winner == HUMAN else 'Draw!')
    return game

print('Bonus game from the updated board:')
play_tic_tac_toe(updated_board, [(1, 0), (1, 2), (2, 1), (2, 2)])

Bonus game from the updated board:
Human plays row 2, column 1:
X | O | X
X | X | .
O | . | O

Computer plays row 3, column 2 (utility 1):
X | O | X
X | X | .
O | O | O

Computer wins!


[['X', 'O', 'X'], ['X', 'X', ' '], ['O', 'O', 'O']]

### Task 3: Minimax vs. Alpha-Beta Pruning - Which Is Faster?

In [4]:
game_tree = {
    'A': ['B', 'C'], 'B': ['D', 'E'], 'C': ['F', 'G'],
    'D': 3, 'E': 5, 'F': 2, 'G': 9
}

def tree3_minimax(node, maximizing_player, tree, visited, values):
    if isinstance(tree[node], int):
        visited.append(node)
        return tree[node]
    child_values = [tree3_minimax(child, not maximizing_player, tree, visited, values)
                    for child in tree[node]]
    values[node] = max(child_values) if maximizing_player else min(child_values)
    return values[node]

def tree3_alpha_beta(node, alpha, beta, maximizing_player, tree, visited):
    if isinstance(tree[node], int):
        visited.append(node)
        return tree[node]
    best = float('-inf') if maximizing_player else float('inf')
    for child in tree[node]:
        value = tree3_alpha_beta(child, alpha, beta, not maximizing_player, tree, visited)
        if maximizing_player:
            best = max(best, value)
            alpha = max(alpha, best)
        else:
            best = min(best, value)
            beta = min(beta, best)
        if alpha >= beta:
            break
    return best

def compare_order(tree, label):
    mm_visited, ab_visited, values = [], [], {}
    mm_value = tree3_minimax('A', True, tree, mm_visited, values)
    ab_value = tree3_alpha_beta('A', float('-inf'), float('inf'), True, tree, ab_visited)
    assert mm_value == ab_value
    skipped = [node for node in mm_visited if node not in ab_visited]
    branch = max(tree['A'], key=lambda node: values[node])
    print(f'{label}: B={values["B"]}, C={values["C"]}, A={mm_value}; selected branch {branch}')
    print(f'Minimax leaves={len(mm_visited)}, alpha-beta leaves={len(ab_visited)}, saved={len(skipped)}')
    print(f'Alpha-beta skipped: {skipped if skipped else "none"}')

compare_order(game_tree, 'Original order')
reordered_tree = {**game_tree, 'A': ['C', 'B']}
compare_order(reordered_tree, 'C explored before B')

Original order: B=3, C=2, A=3; selected branch B
Minimax leaves=4, alpha-beta leaves=3, saved=1
Alpha-beta skipped: ['G']
C explored before B: B=3, C=2, A=3; selected branch B
Minimax leaves=4, alpha-beta leaves=4, saved=0
Alpha-beta skipped: none


### Task 4: Connect Four - Build an Intelligent Game Opponent

Run `play_connect_four()` to play interactively against the computer. The saved game below uses scripted human moves so all outputs remain visible.

In [5]:
import copy

ROWS, COLS = 6, 7
EMPTY, AI, HUMAN = '.', 'O', 'X'

def create_board():
    return [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]

def display_board(board):
    for row in board:
        print(' '.join(row))
    print('1 2 3 4 5 6 7\n')

def get_valid_moves(board):
    return [col for col in range(COLS) if board[0][col] == EMPTY]

def make_move(board, col, player):
    if col not in get_valid_moves(board) or player not in (AI, HUMAN):
        return False
    for row in range(ROWS - 1, -1, -1):
        if board[row][col] == EMPTY:
            board[row][col] = player
            return True
    return False

def windows(board):
    for row in range(ROWS):
        for col in range(COLS - 3):
            yield [board[row][col + i] for i in range(4)]
    for row in range(ROWS - 3):
        for col in range(COLS):
            yield [board[row + i][col] for i in range(4)]
    for row in range(ROWS - 3):
        for col in range(COLS - 3):
            yield [board[row + i][col + i] for i in range(4)]
    for row in range(3, ROWS):
        for col in range(COLS - 3):
            yield [board[row - i][col + i] for i in range(4)]

def check_winner(board):
    for window in windows(board):
        if window.count(AI) == 4:
            return AI
        if window.count(HUMAN) == 4:
            return HUMAN
    return None

def is_terminal(board):
    return check_winner(board) is not None or not get_valid_moves(board)

def evaluate_window(window):
    ai_count, human_count = window.count(AI), window.count(HUMAN)
    if ai_count and human_count:
        return 0
    if ai_count == 4:
        return 100
    if human_count == 4:
        return -100
    if ai_count == 3:
        return 10
    if human_count == 3:
        return -10
    if ai_count == 2:
        return 2
    if human_count == 2:
        return -2
    return 0

def evaluate(board):
    return sum(evaluate_window(window) for window in windows(board))

def minimax(board, depth, alpha, beta, maximizing):
    winner = check_winner(board)
    if winner == AI:
        return 10000 + depth, None
    if winner == HUMAN:
        return -10000 - depth, None
    valid_moves = get_valid_moves(board)
    if not valid_moves:
        return 0, None
    if depth == 0:
        return evaluate(board), None
    ordered_moves = sorted(valid_moves, key=lambda col: (abs(col - COLS // 2), col))
    best_score = float('-inf') if maximizing else float('inf')
    best_col = ordered_moves[0]
    for col in ordered_moves:
        next_board = copy.deepcopy(board)
        make_move(next_board, col, AI if maximizing else HUMAN)
        score, _ = connect4_minimax(next_board, depth - 1, alpha, beta, not maximizing)
        if maximizing and score > best_score or not maximizing and score < best_score:
            best_score, best_col = score, col
        if maximizing:
            alpha = max(alpha, best_score)
        else:
            beta = min(beta, best_score)
        if alpha >= beta:
            break
    return best_score, best_col

def play_connect_four(human_moves=None):
    game = create_board()
    scripted = iter(human_moves) if human_moves is not None else None
    print('New Connect Four game:')
    display_board(game)
    while not is_terminal(game):
        while True:
            try:
                choice = input('Choose a column (1-7): ') if scripted is None else next(scripted)
                col = int(choice) - 1
            except (ValueError, StopIteration):
                if scripted is not None:
                    raise ValueError('Scripted human moves ran out or contain a non-number')
                print('Enter a number from 1 to 7.')
                continue
            if make_move(game, col, HUMAN):
                break
            print(f'Column {col + 1} is unavailable; choose another.')
        print(f'Human plays column {col + 1}:')
        display_board(game)
        if is_terminal(game):
            break
        _, ai_col = connect4_minimax(game, 4, float('-inf'), float('inf'), True)
        make_move(game, ai_col, AI)
        print(f'Computer plays column {ai_col + 1}:')
        display_board(game)
    winner = check_winner(game)
    print('Computer wins!' if winner == AI else 'Human wins!' if winner == HUMAN else 'Draw!')
    return game

connect4_minimax = minimax
empty_board = create_board()
assert len(empty_board) == 6 and all(len(row) == 7 for row in empty_board)
print('Empty board:')
display_board(empty_board)
play_connect_four([3, 2, 4, 1, 5, 6, 7] * 6)

Empty board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
1 2 3 4 5 6 7

New Connect Four game:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
1 2 3 4 5 6 7

Human plays column 3:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X . . . .
1 2 3 4 5 6 7

Computer plays column 5:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X . O . .
1 2 3 4 5 6 7

Human plays column 2:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X X . O . .
1 2 3 4 5 6 7



Computer plays column 4:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X X O O . .
1 2 3 4 5 6 7

Human plays column 4:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. X X O O . .
1 2 3 4 5 6 7



Computer plays column 3:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O X . . .
. X X O O . .
1 2 3 4 5 6 7

Human plays column 1:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O X . . .
X X X O O . .
1 2 3 4 5 6 7



Computer plays column 3:


. . . . . . .
. . . . . . .
. . . . . . .
. . O . . . .
. . O X . . .
X X X O O . .
1 2 3 4 5 6 7

Human plays column 5:
. . . . . . .
. . . . . . .
. . . . . . .
. . O . . . .
. . O X X . .
X X X O O . .
1 2 3 4 5 6 7

Computer plays column 4:
. . . . . . .
. . . . . . .
. . . . . . .
. . O O . . .
. . O X X . .
X X X O O . .
1 2 3 4 5 6 7

Human plays column 6:
. . . . . . .
. . . . . . .
. . . . . . .
. . O O . . .
. . O X X . .
X X X O O X .
1 2 3 4 5 6 7



Computer plays column 5:
. . . . . . .
. . . . . . .
. . . . . . .
. . O O O . .
. . O X X . .
X X X O O X .
1 2 3 4 5 6 7

Human plays column 7:
. . . . . . .
. . . . . . .
. . . . . . .
. . O O O . .
. . O X X . .
X X X O O X X
1 2 3 4 5 6 7



Computer plays column 3:
. . . . . . .
. . . . . . .
. . O . . . .
. . O O O . .
. . O X X . .
X X X O O X X
1 2 3 4 5 6 7

Human plays column 3:
. . . . . . .
. . X . . . .
. . O . . . .
. . O O O . .
. . O X X . .
X X X O O X X
1 2 3 4 5 6 7



Computer plays column 7:
. . . . . . .
. . X . . . .
. . O . . . .
. . O O O . .
. . O X X . O
X X X O O X X
1 2 3 4 5 6 7

Human plays column 2:
. . . . . . .
. . X . . . .
. . O . . . .
. . O O O . .
. X O X X . O
X X X O O X X
1 2 3 4 5 6 7

Computer plays column 2:
. . . . . . .
. . X . . . .
. . O . . . .
. O O O O . .
. X O X X . O
X X X O O X X
1 2 3 4 5 6 7

Computer wins!


[['.', '.', '.', '.', '.', '.', '.'],
 ['.', '.', 'X', '.', '.', '.', '.'],
 ['.', '.', 'O', '.', '.', '.', '.'],
 ['.', 'O', 'O', 'O', 'O', '.', '.'],
 ['.', 'X', 'O', 'X', 'X', '.', 'O'],
 ['X', 'X', 'X', 'O', 'O', 'X', 'X']]

### Task 5: Treasure Guardian - Protect the Treasure Using Minimax

In [6]:
game_tree = {
    'A': ['B', 'C'], 'B': ['D', 'E'], 'C': ['F', 'G'],
    'D': 8, 'E': 3, 'F': 5, 'G': 6
}

def treasure_minimax(node, maximizing, tree, visited, values):
    if isinstance(tree[node], int):
        visited.append(node)
        return tree[node]
    child_values = [treasure_minimax(child, not maximizing, tree, visited, values)
                    for child in tree[node]]
    values[node] = max(child_values) if maximizing else min(child_values)
    return values[node]

def treasure_alpha_beta(node, alpha, beta, maximizing, tree, visited):
    if isinstance(tree[node], int):
        visited.append(node)
        return tree[node]
    best = float('-inf') if maximizing else float('inf')
    for child in tree[node]:
        value = treasure_alpha_beta(child, alpha, beta, not maximizing, tree, visited)
        if maximizing:
            best = max(best, value)
            alpha = max(alpha, best)
        else:
            best = min(best, value)
            beta = min(beta, best)
        if alpha >= beta:
            break
    return best

def compare_treasure(tree, label):
    mm_visited, ab_visited, values = [], [], {}
    mm_score = treasure_minimax('A', True, tree, mm_visited, values)
    ab_score = treasure_alpha_beta('A', float('-inf'), float('inf'), True, tree, ab_visited)
    assert mm_score == ab_score
    action = max(tree['A'], key=lambda node: values[node])
    print(f'{label}: B={values["B"]}, C={values["C"]}, A={values["A"]}')
    print(f'Guardian selects action {action} with guaranteed score {mm_score}')
    print(f'Minimax: score={mm_score}, leaves={len(mm_visited)}; alpha-beta: score={ab_score}, leaves={len(ab_visited)}')

compare_treasure(game_tree, 'Original scores')
modified_tree = {**game_tree, 'E': 7, 'F': 2}
compare_treasure(modified_tree, 'Modified scores (E=7, F=2)')

Original scores: B=3, C=5, A=5
Guardian selects action C with guaranteed score 5
Minimax: score=5, leaves=4; alpha-beta: score=5, leaves=4
Modified scores (E=7, F=2): B=7, C=2, A=7
Guardian selects action B with guaranteed score 7
Minimax: score=7, leaves=4; alpha-beta: score=7, leaves=3


The guardian must account for the intruder choosing the lowest score under each action. This guarantees 5 by choosing C with the original scores. After changing E to 7 and F to 2, the guaranteed score is 7 by choosing B.